# RAG Chatbot Evaluation — Colab (Llama-3.1-8B, HF / vLLM)

Runs the chatbot accuracy eval **off Groq's rate-limited free tier**, so the
judge LLM never gets `429`-throttled mid-run. Accuracy, not speed.

**Metrics (5):**
- `ContextRecall`, `Faithfulness`, `AnswerRelevancy` — ragas (LLM-judged)
- `SemanticSimilarity` — *semantic*: mpnet cosine of answer vs ground truth (added)
- `RougeL` — *lexical*: ROUGE-L F1 of answer vs ground truth (added)

**Model:** judge **and** generator are `meta-llama/Llama-3.1-8B-Instruct` — the open-weights
model behind Groq's `llama-3.1-8b-instant`, run at full precision. Same family as production.

**Pick an engine in the CONFIG cell:**
- `hf_api` — HF Inference Providers. Simplest, real fp16, tiny per-token cost. **Default.**
- `vllm` — local server on the Colab GPU. Free; bf16 on L4/A100, 4-bit on a T4.

> **Clean A/B note:** the committed baseline was judged by *Groq*. For a perfectly controlled
> before/after of the chunking fix, run this notebook twice with the **same** engine: once on the
> cloned (old) KB, once on your rebuilt KB (upload cell). Cross-engine deltas are only *directional*.


## 1. Install dependencies

In [ ]:
%pip install -q "ragas==0.4.3" openai sentence-transformers rouge-score \
                onnxruntime tokenizers langchain-community groq huggingface_hub
# vLLM is installed later, only if you pick ENGINE = "vllm".

## 2. Get the repo (retrieval engine, prompt, ground truth, KB)

In [ ]:
import os
if not os.path.isdir("pose_est_v2"):
    !git clone -q -b test https://github.com/cemmacabales/pose_est_v2.git
%cd pose_est_v2
print("KB on disk:", os.path.getsize("data/knowledge_base.json"), "bytes")

### 2b. (Optional) Upload your rebuilt KB

The clone carries whatever `data/knowledge_base.json` is pushed on the `test` branch.
If your chunking-fix KB **isn't pushed yet**, run this cell and pick your local
`data/knowledge_base.json` to evaluate the fix. Skip otherwise.

In [ ]:
from google.colab import files
import shutil
up = files.upload()
if up:
    name = next(iter(up))
    shutil.move(name, "data/knowledge_base.json")
    print("Replaced data/knowledge_base.json with", name)

## 3. Config

In [ ]:
ENGINE      = "hf_api"      # "hf_api" (HF Inference Providers) | "vllm" (local Colab GPU)
MODEL       = "meta-llama/Llama-3.1-8B-Instruct"   # open weights of Groq's llama-3.1-8b-instant
TOP_K       = 4            # keep at 4 to match the committed baseline apples-to-apples
MAX_TOKENS  = 512
RESULTS_OUT = "eval/rag_results_colab.json"
BASELINE    = "eval/rag_results_source_aware.json"  # committed pre-fix scores (Groq-judged)

## 4. Start the LLM engine

Both branches end up exposing the same OpenAI-compatible `(BASE_URL, API_KEY, MODEL_ID)`,
so generation and judging are identical downstream regardless of engine.

In [ ]:
import os, subprocess, time, urllib.request

if ENGINE == "hf_api":
    from getpass import getpass
    tok = os.environ.get("HF_TOKEN") or getpass("HF token (needs Llama-3.1 access + inference credits): ")
    os.environ["HF_TOKEN"] = tok
    BASE_URL, API_KEY, MODEL_ID = "https://router.huggingface.co/v1", tok, MODEL
    # If the router can't auto-pick a provider, pin one, e.g. MODEL + ":fireworks-ai" or ":together".
    print("HF Inference Providers ->", MODEL_ID)

elif ENGINE == "vllm":
    import torch
    subprocess.run(["pip", "install", "-q", "vllm", "bitsandbytes"], check=True)
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    quant = [] if vram >= 22 else ["--quantization", "bitsandbytes", "--load-format", "bitsandbytes"]
    print(f"GPU ~{vram:.0f}GB -> {'bf16 (closest to Groq)' if not quant else '4-bit bitsandbytes (fits T4)'}")
    MODEL_ID = "llama-3.1-8b-instruct"
    cmd = ["python", "-m", "vllm.entrypoints.openai.api_server", "--model", MODEL,
           "--served-model-name", MODEL_ID, "--dtype", "bfloat16", "--max-model-len", "8192",
           "--gpu-memory-utilization", "0.92",
           "--enable-auto-tool-choice", "--tool-call-parser", "llama3_json"] + quant
    srv = subprocess.Popen(cmd, stdout=open("vllm.log", "w"), stderr=subprocess.STDOUT)
    BASE_URL, API_KEY = "http://localhost:8000/v1", "EMPTY"
    print("Starting vLLM (first run downloads weights; ~3-6 min)... tail vllm.log to watch.")
    for _ in range(180):
        try:
            urllib.request.urlopen("http://localhost:8000/health", timeout=2); print("vLLM ready."); break
        except Exception:
            time.sleep(5)
    else:
        raise RuntimeError("vLLM did not come up — check vllm.log")
else:
    raise ValueError(f"Unknown ENGINE: {ENGINE}")

## 5. Retrieval + faithful chatbot generation

Reuses the repo's real `RetrievalEngine` and `build_system_prompt`, so generated
answers match the deployed chatbot. (`session_chat.llm` requires a `GROQ_API_KEY`
at import time — we set a dummy; no Groq call is ever made here.)

In [ ]:
os.environ.setdefault("GROQ_API_KEY", "unused-on-colab")  # satisfy session_chat.llm import only
from openai import OpenAI, AsyncOpenAI
from session_chat.retrieval import RetrievalEngine
from session_chat.llm import build_system_prompt

gen_client = OpenAI(base_url=BASE_URL, api_key=API_KEY)
retriever  = RetrievalEngine(kb_path="data/knowledge_base.json", model_dir="data/embedding_model")

# Same session stub eval_rag.py uses, so session-specific questions (e.g. sets/reps)
# aren't refused for lack of an exercise list.
SESSION_STUB = {
    "date": "2026-01-15", "duration_seconds": 1800, "overall_form_score_pct": 71,
    "total_exercises_detected": 5,
    "exercises": [
        {"name": "Deep Squat", "duration_seconds": 240, "form_score_pct": 72},
        {"name": "Hurdle Step", "duration_seconds": 210, "form_score_pct": 65},
        {"name": "Inline Lunge", "duration_seconds": 240, "form_score_pct": 70},
        {"name": "Standing Leg Raise", "duration_seconds": 180, "form_score_pct": 68},
        {"name": "Side Lunge", "duration_seconds": 200, "form_score_pct": 75},
    ],
}

def generate(query, contexts):
    chunks = [{"text": c, "source": "knowledge_base", "page": "?", "section_title": ""} for c in contexts]
    sysp = build_system_prompt(SESSION_STUB, chunks)
    r = gen_client.chat.completions.create(
        model=MODEL_ID,
        messages=[{"role": "system", "content": sysp}, {"role": "user", "content": query}],
        temperature=0.0, max_tokens=MAX_TOKENS)
    return r.choices[0].message.content

## 6. ragas metrics (3) — same setup as `eval/eval_rag.py`

In [ ]:
import sys
from types import ModuleType
# ragas hard-imports a vertexai chat model removed from langchain-community 0.3+; stub it out.
_k = "langchain_community.chat_models.vertexai"
if _k not in sys.modules:
    _m = ModuleType(_k); _m.ChatVertexAI = type("ChatVertexAI", (), {}); sys.modules[_k] = _m

from ragas.metrics.collections import AnswerRelevancy, ContextRecall, Faithfulness
from ragas.llms import llm_factory
from ragas.embeddings import HuggingFaceEmbeddings

ragas_llm = llm_factory(MODEL_ID, client=AsyncOpenAI(base_url=BASE_URL, api_key=API_KEY))
ragas_emb = HuggingFaceEmbeddings(model="sentence-transformers/all-MiniLM-L6-v2",
                                  use_api=False, normalize_embeddings=True)
RAGAS_METRICS = [ContextRecall(llm=ragas_llm),
                 Faithfulness(llm=ragas_llm),
                 AnswerRelevancy(llm=ragas_llm, embeddings=ragas_emb)]
print("ragas metrics:", [m.__class__.__name__ for m in RAGAS_METRICS])

## 7. The 2 added metrics (reference-based, deterministic, no LLM)

- **Semantic** — `SemanticSimilarity`: cosine of the answer vs the ground-truth answer in
  `all-mpnet-base-v2` space (a stronger STS model than the system's MiniLM, for a more
  accurate semantic read). "Does the answer *mean* the right thing?"
- **Lexical** — `RougeL`: ROUGE-L F1, longest-common-subsequence word overlap of answer vs
  ground truth. "Does it use the right *words*?"

Both are rate-limit-proof, so they always populate even if a judge call fails.

In [ ]:
from sentence_transformers import SentenceTransformer, util
from rouge_score import rouge_scorer

_sem = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")
def semantic_similarity(resp, ref):
    e = _sem.encode([resp, ref], normalize_embeddings=True)
    return float(util.cos_sim(e[0], e[1]))

_rs = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
def rouge_l(resp, ref):
    return float(_rs.score(ref, resp)["rougeL"].fmeasure)

## 8. Run the evaluation

In [ ]:
import json, time, inspect, math

samples = json.load(open("eval/rag_ground_truth.json"))["samples"]
DELAY = 0  # no Groq TPM cap here; raise if your HF provider rate-limits
rows = []
print(f"Evaluating {len(samples)} samples | engine={ENGINE} model={MODEL_ID} top_k={TOP_K}\n")

for i, item in enumerate(samples):
    q, ref = item["query"], item["ground_truth"]
    ctxs = [c["text"] for c in retriever.search(q, top_k=TOP_K)]
    resp = generate(q, ctxs)
    row = {"query": q,
           "SemanticSimilarity": semantic_similarity(resp, ref),
           "RougeL": rouge_l(resp, ref)}
    payload = {"user_input": q, "retrieved_contexts": ctxs, "response": resp, "reference": ref}
    for metric in RAGAS_METRICS:
        name = metric.__class__.__name__
        req = set(inspect.signature(metric.ascore).parameters) - {"self"}
        try:
            res = metric.batch_score([{k: v for k, v in payload.items() if k in req}])
            val = res[0].value
            row[name] = float(val) if val is not None else float("nan")
        except Exception as e:
            print(f"  [{i}] {name} failed: {e}")
            row[name] = float("nan")
        if DELAY:
            time.sleep(DELAY)
    rows.append(row)
    print(f"  [{i+1:>2}/{len(samples)}] "
          f"CR={row['ContextRecall']:.2f} F={row['Faithfulness']:.2f} AR={row['AnswerRelevancy']:.2f} "
          f"Sem={row['SemanticSimilarity']:.2f} R-L={row['RougeL']:.2f} | {q[:42]}")

json.dump(rows, open(RESULTS_OUT, "w"), indent=2, ensure_ascii=False)
print("\nSaved ->", RESULTS_OUT)

ORDER = ["ContextRecall", "Faithfulness", "AnswerRelevancy", "SemanticSimilarity", "RougeL"]
print("\n=== Aggregate ===")
for k in ORDER:
    vs = [r[k] for r in rows if isinstance(r.get(k), (int, float)) and not math.isnan(r[k])]
    print(f"  {k:18} {sum(vs)/len(vs):.3f}  (n={len(vs)})" if vs else f"  {k:18} n/a")

## 9. Before / after vs the committed baseline

In [ ]:
import json, math

base = {r["query"]: r for r in json.load(open(BASELINE))}
new  = {r["query"]: r for r in rows}

def mean(d, k):
    vs = [r[k] for r in d.values() if isinstance(r.get(k), (int, float)) and not math.isnan(r[k])]
    return (sum(vs) / len(vs), len(vs)) if vs else (float("nan"), 0)

print(f"{'metric':20}{'baseline':>11}{'colab':>11}{'delta':>9}")
for k in ["ContextRecall", "Faithfulness", "AnswerRelevancy"]:
    (bm, _), (nm, _) = mean(base, k), mean(new, k)
    print(f"{k:20}{bm:>11.3f}{nm:>11.3f}{nm-bm:>+9.3f}")
for k in ["SemanticSimilarity", "RougeL"]:
    nm, _ = mean(new, k)
    print(f"{k:20}{'-':>11}{nm:>11.3f}{'new':>9}")

print("\nbaseline judge = Groq llama-3.1-8b-instant | colab judge = Llama-3.1-8B (same family).")
print("Cross-engine deltas are directional. For a strict A/B, run this notebook on the OLD KB too.")